In [ ]:
# ============================================================
# OPENLENS:
# PDF-TEXT EXTRAHIEREN UND OCR-BEDARF ERKENNEN
# ============================================================
#
# Dieser Code:
# - erkennt Projekt- und Dokumentordner automatisch
# - verarbeitet alle heruntergeladenen PDF-Dateien
# - extrahiert vorhandenen Text lokal mit PyMuPDF
# - speichert jedes Dokument als TXT-Datei
# - erkennt wahrscheinliche Scan-PDFs
# - speichert eine vollständige Verarbeitungsübersicht
# - überspringt bereits erfolgreich verarbeitete Dateien
#
# Es wird noch KEINE OCR durchgeführt.
# ============================================================

import re
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display


# ============================================================
# 1. BENÖTIGTE BIBLIOTHEK PRÜFEN UND INSTALLIEREN
# ============================================================

try:
    import fitz  # PyMuPDF

except ImportError:

    print("PyMuPDF ist noch nicht installiert.")
    print("Installation wird jetzt gestartet ...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "pymupdf",
        ]
    )

    import fitz

    print("PyMuPDF wurde erfolgreich installiert.")


# ============================================================
# 2. EINSTELLUNGEN
# ============================================================

# Unterhalb dieses Werts gilt eine Seite als nahezu textlos.
MIN_ZEICHEN_PRO_SEITE = 40

# Wenn mindestens dieser Anteil der Seiten nahezu textlos ist,
# wird das gesamte Dokument als OCR-Kandidat markiert.
OCR_SEITENANTEIL = 0.70

# Nach jeweils so vielen Dokumenten wird der Status gespeichert.
SAVE_EVERY = 10

# Bereits erfolgreich extrahierte Dokumente überspringen.
SKIP_COMPLETED = True


# ============================================================
# 3. PROJEKTORDNER AUTOMATISCH ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()

if ARBEITSORDNER.name.lower() == "datenbank":

    PROJEKTORDNER = ARBEITSORDNER.parent
    DATENBANKORDNER = ARBEITSORDNER

elif (ARBEITSORDNER / "Datenbank").is_dir():

    PROJEKTORDNER = ARBEITSORDNER
    DATENBANKORDNER = ARBEITSORDNER / "Datenbank"

elif ARBEITSORDNER.name.lower() == "dokumente":

    PROJEKTORDNER = ARBEITSORDNER.parent
    DATENBANKORDNER = PROJEKTORDNER / "Datenbank"

else:

    raise FileNotFoundError(
        "\nDer OpenLens-Projektordner konnte nicht erkannt werden.\n\n"
        f"Aktueller Arbeitsordner:\n{ARBEITSORDNER}\n\n"
        "Erwartet wurde entweder:\n"
        "- OpenLens\n"
        "- OpenLens/Datenbank\n"
        "- OpenLens/Dokumente"
    )


DOKUMENTORDNER = PROJEKTORDNER / "Dokumente"
TEXTORDNER = PROJEKTORDNER / "Texte"
OCR_KANDIDATEN_ORDNER = PROJEKTORDNER / "OCR_Kandidaten"

TEXTORDNER.mkdir(
    parents=True,
    exist_ok=True,
)

OCR_KANDIDATEN_ORDNER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. STATUSDATEIEN FESTLEGEN
# ============================================================

STATUS_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_text_extraction_status.csv"
)

STATUS_JSONL = (
    DATENBANKORDNER
    / "fragdenstaat_text_extraction_status.jsonl"
)

FEHLER_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_text_extraction_errors.csv"
)


print("=" * 78)
print("OPENLENS: PDF-TEXTEXTRAKTION")
print("=" * 78)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nProjektordner:")
print(PROJEKTORDNER)

print("\nPDF-Ordner:")
print(DOKUMENTORDNER)

print("\nText-Ordner:")
print(TEXTORDNER)

print("\nStatusdatei:")
print(STATUS_CSV)


# ============================================================
# 5. PDF-DATEIEN SUCHEN
# ============================================================

if not DOKUMENTORDNER.exists():

    raise FileNotFoundError(
        "\nDer Dokumentordner wurde nicht gefunden:\n"
        f"{DOKUMENTORDNER}"
    )


pdf_dateien = sorted(
    [
        pfad
        for pfad in DOKUMENTORDNER.rglob("*")
        if pfad.is_file()
        and pfad.suffix.lower() == ".pdf"
        and not pfad.name.lower().endswith(".part.pdf")
    ]
)


if not pdf_dateien:

    raise FileNotFoundError(
        "\nIm Dokumentordner wurden keine PDF-Dateien gefunden:\n"
        f"{DOKUMENTORDNER}"
    )


print("\nGefundene PDF-Dateien:")
print(f"{len(pdf_dateien):,}")


# ============================================================
# 6. VORHANDENEN STATUS LADEN
# ============================================================

if STATUS_CSV.exists():

    status_df = pd.read_csv(
        STATUS_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    print("\nVorhandener Verarbeitungsstatus wurde geladen.")

else:

    status_df = pd.DataFrame()


if not status_df.empty and "pdf_path" in status_df.columns:

    vorhandener_status = (
        status_df
        .drop_duplicates(
            subset=["pdf_path"],
            keep="last",
        )
        .set_index("pdf_path")
        .to_dict(orient="index")
    )

else:

    vorhandener_status = {}


# ============================================================
# 7. HILFSFUNKTIONEN
# ============================================================

def dokument_id_aus_dateiname(
    dateiname: str,
):
    """
    Liest die FragDenStaat-ID aus Dateinamen wie:
    12345_dokument.pdf
    """

    treffer = re.match(
        r"^(\d+)_",
        str(dateiname),
    )

    if treffer:
        return int(treffer.group(1))

    return None


def text_bereinigen(
    text: str,
) -> str:
    """
    Bereinigt den extrahierten Text vorsichtig.
    """

    if not text:
        return ""

    text = text.replace(
        "\x00",
        "",
    )

    text = text.replace(
        "\r\n",
        "\n",
    )

    text = text.replace(
        "\r",
        "\n",
    )

    # Überflüssige Leerzeichen entfernen
    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    # Mehr als drei Leerzeilen reduzieren
    text = re.sub(
        r"\n{4,}",
        "\n\n\n",
        text,
    )

    return text.strip()


def status_speichern(
    ergebnisse: list[dict],
) -> pd.DataFrame:
    """
    Speichert die vollständige Verarbeitungsübersicht.
    """

    aktueller_df = pd.DataFrame(
        ergebnisse
    )

    if not aktueller_df.empty:

        aktueller_df = (
            aktueller_df
            .drop_duplicates(
                subset=["pdf_path"],
                keep="last",
            )
            .sort_values(
                by=["document_id", "pdf_name"],
                na_position="last",
            )
            .reset_index(drop=True)
        )

    aktueller_df.to_csv(
        STATUS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    aktueller_df.to_json(
        STATUS_JSONL,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    fehler_df = aktueller_df[
        aktueller_df["text_status"].eq("failed")
    ].copy()

    fehler_df.to_csv(
        FEHLER_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    return aktueller_df


def pdf_text_extrahieren(
    pdf_pfad: Path,
) -> dict:
    """
    Extrahiert den Text eines PDFs und bewertet,
    ob OCR wahrscheinlich notwendig ist.
    """

    dokument_id = dokument_id_aus_dateiname(
        pdf_pfad.name
    )

    text_dateiname = (
        f"{pdf_pfad.stem}.txt"
    )

    text_pfad = (
        TEXTORDNER
        / text_dateiname
    )

    ergebnis = {
        "document_id": dokument_id,
        "pdf_name": pdf_pfad.name,
        "pdf_path": str(pdf_pfad.resolve()),
        "pdf_size_bytes": pdf_pfad.stat().st_size,
        "text_path": str(text_pfad.resolve()),
        "num_pages": 0,
        "pages_with_text": 0,
        "pages_without_text": 0,
        "characters_total": 0,
        "characters_per_page": 0.0,
        "textless_page_ratio": 0.0,
        "ocr_required": False,
        "ocr_reason": "",
        "text_status": "processing",
        "text_error": "",
        "processed_at": "",
    }

    dokument = None

    try:

        dokument = fitz.open(
            pdf_pfad
        )

        anzahl_seiten = len(
            dokument
        )

        ergebnis["num_pages"] = anzahl_seiten

        if anzahl_seiten == 0:

            raise ValueError(
                "Das PDF enthält keine Seiten."
            )


        seiten_texte = []
        seiten_mit_text = 0
        seiten_ohne_text = 0


        for seiten_nummer, seite in enumerate(
            dokument,
            start=1,
        ):

            rohtext = seite.get_text(
                "text"
            )

            bereinigter_text = text_bereinigen(
                rohtext
            )

            anzahl_zeichen = len(
                bereinigter_text
            )


            if anzahl_zeichen >= MIN_ZEICHEN_PRO_SEITE:

                seiten_mit_text += 1

            else:

                seiten_ohne_text += 1


            seiten_texte.append(
                "\n".join(
                    [
                        f"===== SEITE {seiten_nummer} =====",
                        "",
                        bereinigter_text,
                        "",
                    ]
                )
            )


        gesamter_text = "\n".join(
            seiten_texte
        ).strip()

        zeichen_gesamt = len(
            gesamter_text
        )

        zeichen_pro_seite = (
            zeichen_gesamt / anzahl_seiten
        )

        textlose_seiten_quote = (
            seiten_ohne_text / anzahl_seiten
        )


        ergebnis["pages_with_text"] = seiten_mit_text
        ergebnis["pages_without_text"] = seiten_ohne_text
        ergebnis["characters_total"] = zeichen_gesamt
        ergebnis["characters_per_page"] = round(
            zeichen_pro_seite,
            2,
        )
        ergebnis["textless_page_ratio"] = round(
            textlose_seiten_quote,
            4,
        )


        # OCR-Entscheidung
        if zeichen_gesamt == 0:

            ergebnis["ocr_required"] = True
            ergebnis["ocr_reason"] = (
                "Kein eingebetteter Text gefunden."
            )

        elif textlose_seiten_quote >= OCR_SEITENANTEIL:

            ergebnis["ocr_required"] = True
            ergebnis["ocr_reason"] = (
                f"{textlose_seiten_quote:.1%} der Seiten "
                "enthalten kaum oder keinen Text."
            )

        elif zeichen_pro_seite < MIN_ZEICHEN_PRO_SEITE:

            ergebnis["ocr_required"] = True
            ergebnis["ocr_reason"] = (
                "Sehr wenig Text pro Seite gefunden."
            )

        else:

            ergebnis["ocr_required"] = False
            ergebnis["ocr_reason"] = (
                "Ausreichend eingebetteter Text vorhanden."
            )


        # Textdatei speichern
        text_pfad.write_text(
            gesamter_text,
            encoding="utf-8",
        )


        # Kleine Hinweisdatei für OCR-Kandidaten speichern
        if ergebnis["ocr_required"]:

            ocr_hinweis = (
                OCR_KANDIDATEN_ORDNER
                / f"{pdf_pfad.stem}.ocr_required.txt"
            )

            ocr_hinweis.write_text(
                "\n".join(
                    [
                        f"PDF: {pdf_pfad}",
                        f"Dokument-ID: {dokument_id}",
                        f"Seiten: {anzahl_seiten}",
                        f"Zeichen insgesamt: {zeichen_gesamt}",
                        f"Textlose Seiten: {seiten_ohne_text}",
                        f"Grund: {ergebnis['ocr_reason']}",
                    ]
                ),
                encoding="utf-8",
            )


        ergebnis["text_status"] = (
            "ocr_required"
            if ergebnis["ocr_required"]
            else "text_extracted"
        )

        ergebnis["processed_at"] = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

        return ergebnis


    except Exception as fehler:

        ergebnis["text_status"] = "failed"

        ergebnis["text_error"] = (
            f"{type(fehler).__name__}: {fehler}"
        )

        ergebnis["processed_at"] = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

        return ergebnis


    finally:

        if dokument is not None:

            dokument.close()


# ============================================================
# 8. BISHERIGE ERGEBNISSE VORBEREITEN
# ============================================================

ergebnisse = []

if not status_df.empty:

    ergebnisse = status_df.to_dict(
        orient="records"
    )


bereits_erfolgreich = {
    str(eintrag.get("pdf_path", "")): eintrag
    for eintrag in ergebnisse
    if eintrag.get("text_status")
    in {
        "text_extracted",
        "ocr_required",
    }
}


zu_verarbeiten = []

for pdf_pfad in pdf_dateien:

    pdf_schluessel = str(
        pdf_pfad.resolve()
    )

    if (
        SKIP_COMPLETED
        and pdf_schluessel in bereits_erfolgreich
    ):

        continue

    zu_verarbeiten.append(
        pdf_pfad
    )


print("\nBereits erfolgreich verarbeitet:")
print(f"{len(bereits_erfolgreich):,}")

print("\nIn diesem Lauf zu verarbeiten:")
print(f"{len(zu_verarbeiten):,}")


# ============================================================
# 9. PDF-TEXTEXTRAKTION STARTEN
# ============================================================

erfolgreich = 0
ocr_kandidaten = 0
fehlgeschlagen = 0


try:

    for position, pdf_pfad in enumerate(
        zu_verarbeiten,
        start=1,
    ):

        print(
            f"[{position}/{len(zu_verarbeiten)}] "
            f"{pdf_pfad.name}",
            end=" ... ",
            flush=True,
        )

        ergebnis = pdf_text_extrahieren(
            pdf_pfad
        )


        # Vorhandenen Eintrag für dieselbe Datei entfernen
        ergebnisse = [
            alter_eintrag
            for alter_eintrag in ergebnisse
            if str(
                alter_eintrag.get(
                    "pdf_path",
                    "",
                )
            )
            != ergebnis["pdf_path"]
        ]

        ergebnisse.append(
            ergebnis
        )


        if ergebnis["text_status"] == "text_extracted":

            erfolgreich += 1
            print("TEXT EXTRAHIERT")

        elif ergebnis["text_status"] == "ocr_required":

            ocr_kandidaten += 1
            print("OCR ERFORDERLICH")

        else:

            fehlgeschlagen += 1
            print(
                f"FEHLER: {ergebnis['text_error']}"
            )


        if position % SAVE_EVERY == 0:

            status_df = status_speichern(
                ergebnisse
            )

            print(
                f"    Zwischenstand nach "
                f"{position} Dokumenten gespeichert."
            )


except KeyboardInterrupt:

    print()
    print("Verarbeitung wurde manuell unterbrochen.")


finally:

    status_df = status_speichern(
        ergebnisse
    )


# ============================================================
# 10. ABSCHLUSSSTATISTIK
# ============================================================

gesamt_text_extrahiert = int(
    status_df["text_status"]
    .eq("text_extracted")
    .sum()
)

gesamt_ocr_required = int(
    status_df["text_status"]
    .eq("ocr_required")
    .sum()
)

gesamt_failed = int(
    status_df["text_status"]
    .eq("failed")
    .sum()
)


print("\n" + "=" * 78)
print("TEXTEXTRAKTION ABGESCHLOSSEN")
print("=" * 78)

print("\nIn diesem Lauf mit vorhandenem Text:")
print(f"{erfolgreich:,}")

print("\nIn diesem Lauf als OCR-Kandidat erkannt:")
print(f"{ocr_kandidaten:,}")

print("\nIn diesem Lauf fehlgeschlagen:")
print(f"{fehlgeschlagen:,}")

print("\nInsgesamt mit Text:")
print(f"{gesamt_text_extrahiert:,}")

print("\nInsgesamt OCR erforderlich:")
print(f"{gesamt_ocr_required:,}")

print("\nInsgesamt fehlgeschlagen:")
print(f"{gesamt_failed:,}")

print("\nGespeicherte Textdateien:")
print(TEXTORDNER)

print("\nOCR-Kandidaten:")
print(OCR_KANDIDATEN_ORDNER)

print("\nVerarbeitungsstatus:")
print(STATUS_CSV)

print("\nFehlerliste:")
print(FEHLER_CSV)


# ============================================================
# 11. STATUSÜBERSICHT ANZEIGEN
# ============================================================

status_uebersicht = (
    status_df["text_status"]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "text_status"
    )
    .reset_index(
        name="anzahl"
    )
)

print("\n" + "=" * 78)
print("STATUSÜBERSICHT")
print("=" * 78)

display(
    status_uebersicht
)


# ============================================================
# 12. OCR-KANDIDATEN ANZEIGEN
# ============================================================

ocr_df = status_df[
    status_df["ocr_required"].eq(True)
].copy()

ocr_anzeige_spalten = [
    spalte
    for spalte in [
        "document_id",
        "pdf_name",
        "num_pages",
        "characters_total",
        "characters_per_page",
        "pages_without_text",
        "textless_page_ratio",
        "ocr_reason",
    ]
    if spalte in ocr_df.columns
]


print("\n" + "=" * 78)
print("ERKANNTE OCR-KANDIDATEN")
print("=" * 78)

if ocr_df.empty:

    print("\nEs wurden keine eindeutigen OCR-Kandidaten erkannt.")

else:

    display(
        ocr_df[
            ocr_anzeige_spalten
        ].head(50)
    )